In [ ]:
# loading the fac documents with questions and answears
from ingest import load_faq_data
documents = load_faq_data()

In [ ]:
# printing the content of the 10th document
documents[10]
# the structure of a single document is as follows: id , course , section ,question, answear

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [ ]:
# initiating an empty list to store documents related to the llm-zoomcamp course
documents_llm = []
# filtering the documents to include only those related to the llm-zoomcamp course 
for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)
# printing the number of documnets related to the llm-zoomcamp course 
len(documents_llm)

103

In [ ]:
# loading the faq documents with questions and answers that only have llm-zoomcamp course
documents = documents_llm

In [ ]:
# select the first documents from the list of documments
doc = documents[0]
# retrieve the id of the documents 
print(doc["id"])
# retrieve the question of the document
print(doc["question"])
# retrieve the answer of the document
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [ ]:
# importing the BaeModel class from the pydantic library to create a data model for questions
from pydantic import BaseModel
# creating a data model for questions using the BaseModel class
class Questions(BaseModel):
    questions: list[str]

In [ ]:
# instructions for generating questions based on the FAQ record
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [13]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
import json
# converting the document to a JSON string to be used as prompt for the OpenAI API
user_prompt = json.dumps(doc)

In [15]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [ ]:
# creating a list of messages to be sent to the OpenAI API, including the data generation instructions and the user prompt 
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [ ]:
# creating the response by openAI client based in the instructions and the user prompt and specifying the model to be used and the text format for the output 
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [ ]:
# retriving the questions generated by the model
response.output_parsed.questions

['I found the course late — can I still join it, or is it too late now?',
 'If I start the course after it already began, can I still get a certificate?',
 'Is it okay to enroll midway through the course and catch up later?',
 'What’s the deadline if I want a certificate after joining late?',
 'Can I submit the project after the submission window closes, or only while submissions are open?']

In [19]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [ ]:
# import the evaluation to structure the result
from evaluation_utils import llm_structured

In [ ]:
# getting the result and usage
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course late — can I still join and take it?', 'Is it too late to start llm-zoomcamp, or can new people still enroll?', 'If I join now, am I still eligible for a certificate?', 'What do I need to do to get a certificate if I’m starting the course late?', 'Are late joiners allowed to submit the project for certification?']


In [ ]:
# retriving the tokens for answear for response 
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=93, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=300)

In [25]:
from evaluation_utils import calc_price

In [26]:
calc_price(usage)

{'input_cost': 0.00015525, 'output_cost': 0.0004185, 'total_cost': 0.00057375}

In [27]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course late — can I still join and take it?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to start llm-zoomcamp, or can new people still enroll?',
  'document': '74eb249bbf'},
 {'question': 'If I join now, am I still eligible for a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to get a certificate if I’m starting the course late?',
  'document': '74eb249bbf'},
 {'question': 'Are late joiners allowed to submit the project for certification?',
  'document': '74eb249bbf'}]

In [28]:
import pandas as pd

In [29]:
pd.DataFrame(records)

,question,document
0,I just found this course late — can I still jo...,74eb249bbf
1,"Is it too late to start llm-zoomcamp, or can n...",74eb249bbf
2,"If I join now, am I still eligible for a certi...",74eb249bbf
3,What do I need to do to get a certificate if I...,74eb249bbf
4,Are late joiners allowed to submit the project...,74eb249bbf


In [30]:
from evaluation_utils import llm_structured_retry

In [31]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [32]:
generate_ground_truth(doc)

([{'question': 'I found this course late — can I still jump in and take it now?',
   'document': '74eb249bbf'},
  {'question': 'Is it okay to start the course after it has already begun, or did I miss the window?',
   'document': '74eb249bbf'},
  {'question': 'Can new students still join the course at this point, and are there any limits?',
   'document': '74eb249bbf'},
  {'question': 'If I enroll now, will I still be eligible for a certificate?',
   'document': '74eb249bbf'},
  {'question': 'What do I need to do if I want the course certificate after joining late?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=99, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=306))

In [33]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [34]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [35]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/103 [00:00<?, ?it/s]

In [36]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

515

In [37]:
ground_truth[10]

{'question': 'Where do I find the live stream link for office hours or workshop sessions?',
 'document': '489dd1c9d9'}

In [38]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.07689675000000003

In [39]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.07689675000000003

In [40]:
df_ground_truth = pd.DataFrame(ground_truth)

In [42]:
df_ground_truth.to_csv("ground_truth.csv", index=False)

In [43]:
len(df_ground_truth)

515